In [1]:
# ============================================================
# STEP 12A — PM2.5 GCCM Train-only preparation
#
# 目标：
# 1. 读取已经冻结的 Formal Train
# 2. 使用 proj_x / proj_y 建 Delaunay graph
# 3. 检查 Ring1 / Ring2 / Ring3
# 4. 对 PM2.5 和 7 个 features 做 Train-only spatial detrending
#
# 暂时不运行 GCCM
# ============================================================

from pathlib import Path
from itertools import combinations
from collections import deque

import json
import hashlib
import numpy as np
import pandas as pd

from scipy.spatial import Delaunay
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components

from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

print("=" * 80)
print("STEP 12A — PM2.5 GCCM TRAIN-ONLY PREPARATION")
print("=" * 80)

# ============================================================
# 0. Paths
# ============================================================

ROOT = Path(
    r"E:\GeoSIMCP_Reproduce\geoconformal-main"
)

DATA_PATH = (
    ROOT
    / "GCCP_PM2.5"
    / "00_data_audit"
    / "china_sites_2018_05_PM2.5_all.csv"
)

OUTPUT_DIR = (
    ROOT
    / "GCCP_PM2.5"
    / "outputs"
)

GCCM_OUTPUT_DIR = (
    OUTPUT_DIR
    / "gccm"
)

GCCM_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

SPLIT_PATH = (
    OUTPUT_DIR
    / "pm25_formal_split_assignment_v1.csv"
)

PROTOCOL_PATH = (
    OUTPUT_DIR
    / "pm25_formal_protocol_v1.json"
)

# ============================================================
# 1. Verify frozen split
# ============================================================

split_df = pd.read_csv(
    SPLIT_PATH,
    encoding="utf-8-sig"
)

with open(
    PROTOCOL_PATH,
    "r",
    encoding="utf-8"
) as f:
    protocol = json.load(f)

hash_text = "\n".join(
    f"{row.original_index}|{row.id}|{row.split}"
    for row in split_df.itertuples()
)

split_hash_now = hashlib.sha256(
    hash_text.encode("utf-8")
).hexdigest()

split_hash_frozen = protocol[
    "split_sha256"
]

print("\nFrozen split hash:")
print(split_hash_frozen)

print("\nCurrent split hash:")
print(split_hash_now)

assert (
    split_hash_now
    == split_hash_frozen
)

print(
    "\n✅ Formal split SHA256 verified"
)

# ============================================================
# 2. Load data
# ============================================================

data = pd.read_csv(
    DATA_PATH,
    encoding="gbk"
)

data = data.rename(
    columns={
        "监测点编码": "id",
        "监测点名称": "name",
        "城市": "city",
        "经度": "lon",
        "纬度": "lat",
    }
)

train_indices = (
    split_df.loc[
        split_df["split"] == "train",
        "original_index"
    ]
    .to_numpy(dtype=int)
)

train = (
    data.iloc[train_indices]
    .copy()
    .reset_index(drop=False)
    .rename(
        columns={
            "index": "original_index"
        }
    )
)

print("\nTrain shape:", train.shape)
print("Train n:", len(train))
print(
    "Unique IDs:",
    train["id"].nunique()
)
print(
    "Unique coordinates:",
    train[
        ["proj_x", "proj_y"]
    ]
    .drop_duplicates()
    .shape[0]
)

assert len(train) == 990
assert train["id"].nunique() == 990

# ============================================================
# 3. Delaunay graph
#
# proj_x / proj_y:
# 已经在 STEP 4 验证其局部距离结构
# 与 Haversine 高度一致。
#
# GCCM 需要平面邻接结构，
# 所以这里用 projected coordinates。
# ============================================================

XY = train[
    ["proj_x", "proj_y"]
].to_numpy(dtype=float)

tri = Delaunay(
    XY
)

triangles = tri.simplices

edge_set = set()

for triangle in triangles:

    for a, b in combinations(
        triangle,
        2
    ):

        u = int(min(a, b))
        v = int(max(a, b))

        edge_set.add(
            (u, v)
        )

edges = sorted(
    edge_set
)

print("\n" + "=" * 80)
print("DELAUNAY GRAPH")
print("=" * 80)

print(
    "Sites      :",
    len(train)
)

print(
    "Triangles  :",
    len(triangles)
)

print(
    "Edges      :",
    len(edges)
)

mean_degree = (
    2 * len(edges)
    / len(train)
)

print(
    "Mean degree:",
    mean_degree
)

# ============================================================
# 4. Connectivity
# ============================================================

rows = []
cols = []

for u, v in edges:

    rows.extend(
        [u, v]
    )

    cols.extend(
        [v, u]
    )

A = csr_matrix(
    (
        np.ones(
            len(rows),
            dtype=int
        ),
        (
            rows,
            cols
        )
    ),
    shape=(
        len(train),
        len(train)
    )
)

n_components, labels = (
    connected_components(
        A,
        directed=False
    )
)

print(
    "\nConnected components:",
    n_components
)

assert n_components == 1

print(
    "✅ Delaunay graph is connected"
)

# ============================================================
# 5. Adjacency list
# ============================================================

neighbors = [
    set()
    for _ in range(
        len(train)
    )
]

for u, v in edges:

    neighbors[u].add(v)
    neighbors[v].add(u)

degree = np.array(
    [
        len(nbr)
        for nbr in neighbors
    ]
)

print("\nDegree:")
print(
    pd.Series(
        degree
    ).describe()
)

# ============================================================
# 6. Ring1 / Ring2 / Ring3
#
# Ring k =
# graph shortest-path distance exactly k
#
# E = 3 requires first 3 spatial rings.
# ============================================================

ring_sizes = {
    1: [],
    2: [],
    3: [],
}

ring_members = {}

for start in range(
    len(train)
):

    visited = {
        start
    }

    frontier = {
        start
    }

    ring_members[start] = {}

    for ring in [
        1,
        2,
        3
    ]:

        next_frontier = set()

        for node in frontier:

            next_frontier.update(
                neighbors[node]
            )

        next_frontier -= visited

        ring_members[
            start
        ][ring] = sorted(
            next_frontier
        )

        ring_sizes[
            ring
        ].append(
            len(
                next_frontier
            )
        )

        visited.update(
            next_frontier
        )

        frontier = (
            next_frontier
        )

# ============================================================
# 7. Ring diagnostics
# ============================================================

ring_rows = []

for ring in [
    1,
    2,
    3
]:

    arr = np.asarray(
        ring_sizes[ring]
    )

    ring_rows.append({
        "ring":
            ring,

        "mean_size":
            arr.mean(),

        "min_size":
            arr.min(),

        "median_size":
            np.median(arr),

        "max_size":
            arr.max(),

        "empty_rings":
            np.sum(
                arr == 0
            ),
    })

ring_df = pd.DataFrame(
    ring_rows
)

print("\n" + "=" * 80)
print("RING DIAGNOSTICS")
print("=" * 80)

display(
    ring_df
)

if (
    ring_df[
        "empty_rings"
    ].sum()
    > 0
):

    raise RuntimeError(
        "At least one Ring1/2/3 is empty. "
        "Do not run GCCM yet."
    )

print(
    "✅ Ring1 / Ring2 / Ring3 "
    "all valid for every Train site"
)

# ============================================================
# 8. Train-only spatial detrending
#
# v = beta0 + beta1*proj_x + beta2*proj_y + residual
#
# GCCM 使用 residual，
# coordinates themselves only define spatial topology.
# ============================================================

FEATURES = [
    "dem",
    "sp",
    "tp",
    "blh",
    "r",
    "aod_sat",
    "ndvi",
]

TARGET = "PM2.5"

GCCM_VARIABLES = (
    [TARGET]
    + FEATURES
)

detrended = pd.DataFrame(
    index=train.index
)

detrend_rows = []

for variable in GCCM_VARIABLES:

    y_raw = train[
        variable
    ].to_numpy(dtype=float)

    model = LinearRegression()

    model.fit(
        XY,
        y_raw
    )

    trend = model.predict(
        XY
    )

    residual = (
        y_raw
        - trend
    )

    detrended[
        variable
    ] = residual

    detrend_rows.append({
        "variable":
            variable,

        "trend_R2":
            r2_score(
                y_raw,
                trend
            ),

        "raw_mean":
            np.mean(
                y_raw
            ),

        "raw_std":
            np.std(
                y_raw,
                ddof=1
            ),

        "detrended_mean":
            np.mean(
                residual
            ),

        "detrended_std":
            np.std(
                residual,
                ddof=1
            ),

        "coef_x":
            model.coef_[0],

        "coef_y":
            model.coef_[1],

        "intercept":
            model.intercept_,
    })


detrend_df = pd.DataFrame(
    detrend_rows
)

print("\n" + "=" * 80)
print("TRAIN-ONLY DETREND DIAGNOSTICS")
print("=" * 80)

display(
    detrend_df
)

# ============================================================
# 9. Prepare GCCM input
# ============================================================

gccm_input = pd.DataFrame({
    "local_index_0":
        np.arange(
            len(train)
        ),

    # R-friendly one-based index
    "site_index_1":
        np.arange(
            1,
            len(train) + 1
        ),

    "original_index":
        train[
            "original_index"
        ].to_numpy(),

    "id":
        train[
            "id"
        ].astype(str)
        .to_numpy(),

    "proj_x":
        train[
            "proj_x"
        ].to_numpy(),

    "proj_y":
        train[
            "proj_y"
        ].to_numpy(),
})

for variable in GCCM_VARIABLES:

    gccm_input[
        f"{variable}_raw"
    ] = (
        train[
            variable
        ].to_numpy()
    )

    gccm_input[
        f"{variable}_detrended"
    ] = (
        detrended[
            variable
        ].to_numpy()
    )

# ============================================================
# 10. Edge table
# ============================================================

edge_df = pd.DataFrame(
    edges,
    columns=[
        "u0",
        "v0"
    ]
)

edge_df[
    "u1"
] = (
    edge_df[
        "u0"
    ] + 1
)

edge_df[
    "v1"
] = (
    edge_df[
        "v0"
    ] + 1
)

edge_df[
    "u_id"
] = (
    train.iloc[
        edge_df["u0"]
    ]["id"]
    .astype(str)
    .to_numpy()
)

edge_df[
    "v_id"
] = (
    train.iloc[
        edge_df["v0"]
    ]["id"]
    .astype(str)
    .to_numpy()
)

# ============================================================
# 11. Save
# ============================================================

INPUT_PATH = (
    GCCM_OUTPUT_DIR
    / "pm25_gccm_train_detrended_v1.csv"
)

EDGE_PATH = (
    GCCM_OUTPUT_DIR
    / "pm25_gccm_delaunay_edges_v1.csv"
)

RING_PATH = (
    GCCM_OUTPUT_DIR
    / "pm25_gccm_ring_diagnostics_v1.csv"
)

DETREND_PATH = (
    GCCM_OUTPUT_DIR
    / "pm25_gccm_detrend_diagnostics_v1.csv"
)

gccm_input.to_csv(
    INPUT_PATH,
    index=False,
    encoding="utf-8-sig"
)

edge_df.to_csv(
    EDGE_PATH,
    index=False,
    encoding="utf-8-sig"
)

ring_df.to_csv(
    RING_PATH,
    index=False,
    encoding="utf-8-sig"
)

detrend_df.to_csv(
    DETREND_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("\n" + "=" * 80)
print("SAVED")
print("=" * 80)

print(INPUT_PATH)
print(EDGE_PATH)
print(RING_PATH)
print(DETREND_PATH)

print("\n✅ STEP 12A 完成")
print("✅ 仅 Formal Train 用于 GCCM preparation")
print("⚠️ 还没有运行 GCCM")

STEP 12A — PM2.5 GCCM TRAIN-ONLY PREPARATION

Frozen split hash:
785436f18bcc08a5a6b94916d3e544af0e0706e959b7c930300f7a91bddf04f3

Current split hash:
785436f18bcc08a5a6b94916d3e544af0e0706e959b7c930300f7a91bddf04f3

✅ Formal split SHA256 verified

Train shape: (990, 25)
Train n: 990
Unique IDs: 990
Unique coordinates: 990

DELAUNAY GRAPH
Sites      : 990
Triangles  : 1961
Edges      : 2950
Mean degree: 5.959595959595959

Connected components: 1
✅ Delaunay graph is connected

Degree:
count    990.000000
mean       5.959596
std        1.700646
min        3.000000
25%        5.000000
50%        6.000000
75%        7.000000
max       12.000000
dtype: float64

RING DIAGNOSTICS


,ring,mean_size,min_size,median_size,max_size,empty_rings
0,1,5.959596,3,6.0,12,0
1,2,14.242424,5,14.0,40,0
2,3,25.626263,9,25.0,69,0


✅ Ring1 / Ring2 / Ring3 all valid for every Train site

TRAIN-ONLY DETREND DIAGNOSTICS


,variable,trend_R2,raw_mean,raw_std,detrended_mean,detrended_std,coef_x,coef_y,intercept
0,PM2.5,0.021712,33.172300,13.103830,1.837363e-15,12.960793,-1.782086e-06,2.131788e-06,34.726282
1,dem,0.480522,393.035354,662.123121,9.003079e-14,477.224007,-7.099904e-04,4.537201e-05,1470.810484
2,sp,0.507096,96027.597266,7438.089495,1.190611e-11,5222.068829,8.192888e-03,-5.145490e-04,83585.537702
3,tp,0.533324,0.002208,0.001346,-1.401797e-19,0.000919,6.445628e-10,-1.186081e-09,0.001883
4,blh,0.155387,244.212708,124.676573,1.446923e-14,114.581271,-7.540239e-05,-8.071887e-07,361.884173
5,r,0.739557,65.490947,15.847975,-7.579123e-15,8.087797,1.287055e-05,-1.489691e-05,53.982985
6,aod_sat,0.175855,0.566491,0.171904,4.037175e-18,0.156059,1.036021e-07,-4.418379e-08,0.430652
7,ndvi,0.052561,3322.096970,1162.894927,-3.674726e-15,1131.920991,3.066614e-05,-3.380363e-04,3467.282872



SAVED
E:\GeoSIMCP_Reproduce\geoconformal-main\GCCP_PM2.5\outputs\gccm\pm25_gccm_train_detrended_v1.csv
E:\GeoSIMCP_Reproduce\geoconformal-main\GCCP_PM2.5\outputs\gccm\pm25_gccm_delaunay_edges_v1.csv
E:\GeoSIMCP_Reproduce\geoconformal-main\GCCP_PM2.5\outputs\gccm\pm25_gccm_ring_diagnostics_v1.csv
E:\GeoSIMCP_Reproduce\geoconformal-main\GCCP_PM2.5\outputs\gccm\pm25_gccm_detrend_diagnostics_v1.csv

✅ STEP 12A 完成
✅ 仅 Formal Train 用于 GCCM preparation
⚠️ 还没有运行 GCCM


In [2]:
# ============================================================
# STEP 12B — Run PM2.5 Train-only GCCM in R
#
# E = 3
# tau = 1
# b = 4
# L = [10, 50, 100, 200, 400, 988]
#
# Both directions are saved:
#
# cause_xmap_effect:
#     X manifold reconstructs PM2.5
#
# effect_xmap_cause:
#     PM2.5 manifold reconstructs X
#
# For pre-specified X -> PM2.5 causal strength,
# we will later use effect_xmap_cause.
# ============================================================

from pathlib import Path
import subprocess
import textwrap
import pandas as pd

ROOT = Path(
    r"E:\GeoSIMCP_Reproduce\geoconformal-main"
)

GCCM_DIR = (
    ROOT
    / "GCCP_PM2.5"
    / "03_gccm"
)

GCCM_OUTPUT_DIR = (
    ROOT
    / "GCCP_PM2.5"
    / "outputs"
    / "gccm"
)

INPUT_PATH = (
    GCCM_OUTPUT_DIR
    / "pm25_gccm_train_detrended_v1.csv"
)

EDGE_PATH = (
    GCCM_OUTPUT_DIR
    / "pm25_gccm_delaunay_edges_v1.csv"
)

# 使用 Seattle 已经验证过的 GCCM 源码
GCCM_SOURCE = (
    ROOT
    / "GCCP_Seattle"
    / "02_gccm"
    / "GCCM_updated 2"
    / "R"
    / "GCCM"
    / "GCCM4Lattice.r"
)

RSCRIPT = Path(
    r"C:\Program Files\R\R-4.6.1\bin\Rscript.exe"
)

R_RUNNER = (
    GCCM_DIR
    / "run_pm25_gccm_train_only_v1.R"
)

RAW_OUTPUT = (
    GCCM_OUTPUT_DIR
    / "pm25_gccm_raw_rho_v1.csv"
)

MEAN_OUTPUT = (
    GCCM_OUTPUT_DIR
    / "pm25_gccm_mean_curves_v1.csv"
)

# ------------------------------------------------------------
# 1. 路径检查
# ------------------------------------------------------------

print("=" * 80)
print("STEP 12B — PM2.5 TRAIN-ONLY GCCM")
print("=" * 80)

paths = {
    "Rscript": RSCRIPT,
    "GCCM source": GCCM_SOURCE,
    "Train input": INPUT_PATH,
    "Delaunay edges": EDGE_PATH,
}

for name, p in paths.items():
    print(
        f"{name:<16s}: "
        f"{p.exists()}  {p}"
    )

assert RSCRIPT.exists()
assert GCCM_SOURCE.exists()
assert INPUT_PATH.exists()
assert EDGE_PATH.exists()

# ------------------------------------------------------------
# 2. 转成 R 最稳定的 forward-slash 路径
# ------------------------------------------------------------

def rpath(p):
    return Path(p).resolve().as_posix()

# ------------------------------------------------------------
# 3. 生成 R runner
# ------------------------------------------------------------

r_code = f'''
options(warn = 1)

suppressPackageStartupMessages({{
    library(parallel)
    library(foreach)
    library(doParallel)
}})

cat(
    "============================================================\\n"
)
cat(
    "PM2.5 TRAIN-ONLY GCCM\\n"
)
cat(
    "============================================================\\n"
)

# ============================================================
# Paths
# ============================================================

input_path <- "{rpath(INPUT_PATH)}"
edge_path  <- "{rpath(EDGE_PATH)}"
source_path <- "{rpath(GCCM_SOURCE)}"

raw_output  <- "{rpath(RAW_OUTPUT)}"
mean_output <- "{rpath(MEAN_OUTPUT)}"

source(source_path)

cat("GCCM source loaded.\\n")

# ============================================================
# Read Train-only data
# ============================================================

dat <- read.csv(
    input_path,
    check.names = FALSE,
    fileEncoding = "UTF-8-BOM"
)

edges <- read.csv(
    edge_path,
    check.names = FALSE,
    fileEncoding = "UTF-8-BOM"
)

n <- nrow(dat)

cat("Train sites:", n, "\\n")

if (n != 990) {{
    stop(
        paste0(
            "Expected 990 Train sites, got ",
            n
        )
    )
}}

# ============================================================
# Build neighborMatrix
#
# GCCM uses 1-based integer site indices.
# ============================================================

neighborMatrix <- vector(
    "list",
    n
)

for (i in seq_len(n)) {{
    neighborMatrix[[i]] <- integer(0)
}}

for (i in seq_len(nrow(edges))) {{

    u <- as.integer(
        edges$u1[i]
    )

    v <- as.integer(
        edges$v1[i]
    )

    neighborMatrix[[u]] <- c(
        neighborMatrix[[u]],
        v
    )

    neighborMatrix[[v]] <- c(
        neighborMatrix[[v]],
        u
    )
}}

neighborMatrix <- lapply(
    neighborMatrix,
    function(x) {{
        sort(
            unique(
                as.integer(x)
            )
        )
    }}
)

degree <- lengths(
    neighborMatrix
)

cat(
    "Mean degree:",
    mean(degree),
    "\\n"
)

cat(
    "Minimum degree:",
    min(degree),
    "\\n"
)

if (any(degree == 0)) {{
    stop(
        "At least one site has no Delaunay neighbor."
    )
}}

# ============================================================
# Frozen GCCM settings
# ============================================================

E <- 3
tau <- 1
b <- 4

lib <- c(
    1,
    n
)

pred <- c(
    1,
    n
)

max_lib_expected <- (
    n
    - (E - 1) * tau
)

lib_sizes <- c(
    10,
    50,
    100,
    200,
    400,
    max_lib_expected
)

cat("\\n")
cat(
    "E                =",
    E,
    "\\n"
)

cat(
    "tau              =",
    tau,
    "\\n"
)

cat(
    "b                =",
    b,
    "\\n"
)

cat(
    "max library size =",
    max_lib_expected,
    "\\n"
)

cat(
    "library sizes    =",
    paste(
        lib_sizes,
        collapse = ", "
    ),
    "\\n"
)

if (max_lib_expected != 988) {{
    stop(
        "Expected max library size = 988."
    )
}}

# ============================================================
# Variables
# ============================================================

features <- c(
    "dem",
    "sp",
    "tp",
    "blh",
    "r",
    "aod_sat",
    "ndvi"
)

effect_name <- "PM2.5"

effect_col <- paste0(
    effect_name,
    "_detrended"
)

y <- as.numeric(
    dat[[effect_col]]
)

if (any(!is.finite(y))) {{
    stop(
        "PM2.5 detrended target contains non-finite values."
    )
}}

# ============================================================
# Helper: aggregate GCCM rho for each library size
#
# Match the original polygon workflow:
#
# max(0, mean(rho, na.rm=TRUE))
# ============================================================

mean_curve <- function(
    res,
    feature_name,
    direction_name
) {{

    split_res <- split(
        res,
        res$L
    )

    rho_mean <- sapply(
        split_res,
        function(z) {{

            temp <- mean(
                z$rho,
                na.rm = TRUE
            )

            max(
                0,
                temp
            )
        }}
    )

    out <- data.frame(
        feature = feature_name,
        direction = direction_name,
        lib_size = as.integer(
            names(rho_mean)
        ),
        rho = as.numeric(
            rho_mean
        )
    )

    out <- out[
        order(
            out$lib_size
        ),
    ]

    rownames(out) <- NULL

    return(out)
}}

# ============================================================
# Run
# ============================================================

all_raw <- list()
all_mean <- list()

raw_counter <- 1
mean_counter <- 1

# No random sampling is used in GCCMSingle4Lattice,
# but fixing seed makes the runner fully explicit.
set.seed(42)

for (feature_name in features) {{

    cat("\\n")
    cat(
        "------------------------------------------------------------\\n"
    )

    cat(
        "Feature:",
        feature_name,
        "\\n"
    )

    x_col <- paste0(
        feature_name,
        "_detrended"
    )

    x <- as.numeric(
        dat[[x_col]]
    )

    if (any(!is.finite(x))) {{
        stop(
            paste(
                feature_name,
                "contains non-finite values"
            )
        )
    }}

    # --------------------------------------------------------
    # Embedding from candidate cause X
    # --------------------------------------------------------

    embed_x <- generateEmbedings(
        neighborMatrix,
        x,
        E
    )

    # --------------------------------------------------------
    # Embedding from effect PM2.5
    # --------------------------------------------------------

    embed_y <- generateEmbedings(
        neighborMatrix,
        y,
        E
    )

    if (
        any(!is.finite(embed_x))
        ||
        any(!is.finite(embed_y))
    ) {{
        stop(
            paste(
                "Non-finite embedding for",
                feature_name
            )
        )
    }}

    # --------------------------------------------------------
    # Direction 1:
    # X manifold reconstructs PM2.5
    # --------------------------------------------------------

    cat(
        "  Running cause_xmap_effect ...\\n"
    )

    x_xmap_y <- GCCMLattice(
        embed_x,
        y,
        lib_sizes,
        lib,
        pred,
        E,
        tau = tau,
        b = b,
        cores = 6
    )

    x_xmap_y$feature <- feature_name
    x_xmap_y$direction <- "cause_xmap_effect"

    all_raw[[raw_counter]] <- x_xmap_y

    raw_counter <- raw_counter + 1

    all_mean[[mean_counter]] <- mean_curve(
        x_xmap_y,
        feature_name,
        "cause_xmap_effect"
    )

    mean_counter <- mean_counter + 1

    # --------------------------------------------------------
    # Direction 2:
    # PM2.5 manifold reconstructs X
    #
    # THIS is the pre-specified direction used later
    # for X -> PM2.5 causal strength.
    # --------------------------------------------------------

    cat(
        "  Running effect_xmap_cause ...\\n"
    )

    y_xmap_x <- GCCMLattice(
        embed_y,
        x,
        lib_sizes,
        lib,
        pred,
        E,
        tau = tau,
        b = b,
        cores = 6
    )

    y_xmap_x$feature <- feature_name
    y_xmap_x$direction <- "effect_xmap_cause"

    all_raw[[raw_counter]] <- y_xmap_x

    raw_counter <- raw_counter + 1

    all_mean[[mean_counter]] <- mean_curve(
        y_xmap_x,
        feature_name,
        "effect_xmap_cause"
    )

    mean_counter <- mean_counter + 1

    cat(
        "  Finished:",
        feature_name,
        "\\n"
    )
}}

# ============================================================
# Combine and save
# ============================================================

raw_df <- do.call(
    rbind,
    all_raw
)

mean_df <- do.call(
    rbind,
    all_mean
)

raw_df <- raw_df[
    ,
    c(
        "feature",
        "direction",
        "L",
        "rho"
    )
]

names(raw_df)[
    names(raw_df) == "L"
] <- "lib_size"

mean_df <- mean_df[
    order(
        mean_df$feature,
        mean_df$direction,
        mean_df$lib_size
    ),
]

rownames(raw_df) <- NULL
rownames(mean_df) <- NULL

write.csv(
    raw_df,
    raw_output,
    row.names = FALSE,
    fileEncoding = "UTF-8"
)

write.csv(
    mean_df,
    mean_output,
    row.names = FALSE,
    fileEncoding = "UTF-8"
)

cat("\\n")
cat(
    "============================================================\\n"
)

cat(
    "GCCM COMPLETE\\n"
)

cat(
    "============================================================\\n"
)

cat(
    "Raw rows :",
    nrow(raw_df),
    "\\n"
)

cat(
    "Mean rows:",
    nrow(mean_df),
    "\\n"
)

cat(
    "Raw output :",
    raw_output,
    "\\n"
)

cat(
    "Mean output:",
    mean_output,
    "\\n"
)
'''

R_RUNNER.write_text(
    textwrap.dedent(r_code),
    encoding="utf-8"
)

print("\nR runner 已生成:")
print(R_RUNNER)

# ------------------------------------------------------------
# 4. Execute R
# ------------------------------------------------------------

print("\n开始运行 GCCM ...")
print(
    "这一步计算量明显高于前面的 CP/GeoSIMCP。"
)

result = subprocess.run(
    [
        str(RSCRIPT),
        str(R_RUNNER)
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True,
    encoding="utf-8",
    errors="replace"
)

print("\n" + "=" * 80)
print("R STDOUT")
print("=" * 80)

print(
    result.stdout
)

if result.stderr.strip():

    print("\n" + "=" * 80)
    print("R STDERR / WARNINGS")
    print("=" * 80)

    print(
        result.stderr
    )

print(
    "\nR return code:",
    result.returncode
)

if result.returncode != 0:
    raise RuntimeError(
        "PM2.5 GCCM R runner failed. "
        "请把上面的 R STDERR 发给我。"
    )

assert RAW_OUTPUT.exists()
assert MEAN_OUTPUT.exists()

print("\n✅ STEP 12B GCCM 运行成功")

# ------------------------------------------------------------
# 5. Quick output audit
# ------------------------------------------------------------

mean_curve_df = pd.read_csv(
    MEAN_OUTPUT
)

print("\nMean curve shape:")
print(
    mean_curve_df.shape
)

print("\nLibrary sizes:")
print(
    sorted(
        mean_curve_df[
            "lib_size"
        ].unique()
    )
)

print("\nDirection counts:")
print(
    mean_curve_df[
        "direction"
    ].value_counts()
)

print("\nMean curves:")
display(
    mean_curve_df
)

STEP 12B — PM2.5 TRAIN-ONLY GCCM
Rscript         : True  C:\Program Files\R\R-4.6.1\bin\Rscript.exe
GCCM source     : True  E:\GeoSIMCP_Reproduce\geoconformal-main\GCCP_Seattle\02_gccm\GCCM_updated 2\R\GCCM\GCCM4Lattice.r
Train input     : True  E:\GeoSIMCP_Reproduce\geoconformal-main\GCCP_PM2.5\outputs\gccm\pm25_gccm_train_detrended_v1.csv
Delaunay edges  : True  E:\GeoSIMCP_Reproduce\geoconformal-main\GCCP_PM2.5\outputs\gccm\pm25_gccm_delaunay_edges_v1.csv

R runner 已生成:
E:\GeoSIMCP_Reproduce\geoconformal-main\GCCP_PM2.5\03_gccm\run_pm25_gccm_train_only_v1.R

开始运行 GCCM ...
这一步计算量明显高于前面的 CP/GeoSIMCP。

R STDOUT
PM2.5 TRAIN-ONLY GCCM
GCCM source loaded.
Train sites: 990 
Mean degree: 5.959596 
Minimum degree: 3 

E                = 3 
tau              = 1 
b                = 4 
max library size = 988 
library sizes    = 10, 50, 100, 200, 400, 988 

------------------------------------------------------------
Feature: dem 
  Running cause_xmap_effect ...
  Running effect_xmap_cause ...
 

,feature,direction,lib_size,rho
0,aod_sat,cause_xmap_effect,10,0.140372
1,aod_sat,cause_xmap_effect,50,0.279763
2,aod_sat,cause_xmap_effect,100,0.320401
3,aod_sat,cause_xmap_effect,200,0.358034
4,aod_sat,cause_xmap_effect,400,0.400520
...,...,...,...,...
79,tp,effect_xmap_cause,50,0.055519
80,tp,effect_xmap_cause,100,0.086514
81,tp,effect_xmap_cause,200,0.147823
82,tp,effect_xmap_cause,400,0.197761


In [3]:
# ============================================================
# STEP 12C — PM2.5 GCCM convergence diagnostics
#
# Frozen Seattle-aligned rules:
#
# 1. gain > 0
# 2. Spearman rho(L) >= 0.60
# 3. terminal rho two-sided Pearson p < 0.05
# 4. terminal rho Fisher-z 95% CI lower > 0
#
# If all pass:
#     causal_strength = terminal rho
# else:
#     causal_strength = 0
#
# Then normalize causal weights.
#
# Direction used for pre-specified X -> PM2.5:
#     effect_xmap_cause
#     PM2.5 manifold reconstructs X
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd

from scipy.stats import (
    spearmanr,
    t as student_t,
    norm
)

print("=" * 80)
print("STEP 12C — PM2.5 GCCM CAUSAL WEIGHTS")
print("=" * 80)

# ============================================================
# 0. Paths
# ============================================================

ROOT = Path(
    r"E:\GeoSIMCP_Reproduce\geoconformal-main"
)

GCCM_OUTPUT_DIR = (
    ROOT
    / "GCCP_PM2.5"
    / "outputs"
    / "gccm"
)

MEAN_OUTPUT = (
    GCCM_OUTPUT_DIR
    / "pm25_gccm_mean_curves_v1.csv"
)

RAW_OUTPUT = (
    GCCM_OUTPUT_DIR
    / "pm25_gccm_raw_rho_v1.csv"
)

mean_df = pd.read_csv(
    MEAN_OUTPUT
)

raw_df = pd.read_csv(
    RAW_OUTPUT
)

print("Mean rows:", len(mean_df))
print("Raw rows :", len(raw_df))

# ============================================================
# 1. Frozen settings
# ============================================================

FEATURES = [
    "dem",
    "sp",
    "tp",
    "blh",
    "r",
    "aod_sat",
    "ndvi",
]

DIRECTION = "effect_xmap_cause"

LIB_SIZES = np.array(
    [10, 50, 100, 200, 400, 988],
    dtype=float
)

L_SMALL = 10
L_MAX = 988

SPEARMAN_THRESHOLD = 0.60
P_THRESHOLD = 0.05
CI_LEVEL = 0.95

# With E=3, tau=1 and n=990:
#
# pred_indices are positions 3..990
# => N = 988 terminal predictions
N_TERMINAL = 988

print("\nFrozen criteria:")
print("gain > 0")
print("Spearman >=", SPEARMAN_THRESHOLD)
print("p <", P_THRESHOLD)
print("CI lower > 0")
print("Terminal N =", N_TERMINAL)

# ============================================================
# 2. Pearson correlation significance
# ============================================================

def pearson_r_pvalue(r, n):
    """
    Two-sided Pearson correlation t-test.

    t = r * sqrt((n-2)/(1-r^2))
    df = n-2
    """
    r = float(r)

    if not np.isfinite(r):
        return np.nan

    # protect r=+-1
    r = np.clip(
        r,
        -0.999999999999,
        0.999999999999
    )

    df = n - 2

    t_stat = (
        r
        * np.sqrt(
            df / (1 - r**2)
        )
    )

    p = (
        2
        * student_t.sf(
            abs(t_stat),
            df=df
        )
    )

    return float(p)

# ============================================================
# 3. Fisher-z confidence interval
# ============================================================

def fisher_z_ci(
    r,
    n,
    confidence=0.95
):
    """
    Fisher z transform:
        z = arctanh(r)
        SE = 1/sqrt(n-3)
    """
    r = float(r)

    r = np.clip(
        r,
        -0.999999999999,
        0.999999999999
    )

    z = np.arctanh(r)

    se = 1 / np.sqrt(
        n - 3
    )

    alpha_ci = (
        1 - confidence
    )

    zcrit = norm.ppf(
        1 - alpha_ci / 2
    )

    lower_z = (
        z - zcrit * se
    )

    upper_z = (
        z + zcrit * se
    )

    lower = np.tanh(
        lower_z
    )

    upper = np.tanh(
        upper_z
    )

    return (
        float(lower),
        float(upper)
    )

# ============================================================
# 4. Build diagnostics
# ============================================================

diagnostic_rows = []

for feature in FEATURES:

    curve = (
        mean_df[
            (mean_df["feature"] == feature)
            &
            (mean_df["direction"] == DIRECTION)
        ]
        .sort_values("lib_size")
        .copy()
    )

    if len(curve) != 6:
        raise RuntimeError(
            f"{feature}: expected 6 library sizes, "
            f"got {len(curve)}"
        )

    L = curve[
        "lib_size"
    ].to_numpy(dtype=float)

    rho_curve = curve[
        "rho"
    ].to_numpy(dtype=float)

    assert np.array_equal(
        L,
        LIB_SIZES
    )

    rho_L10 = float(
        curve.loc[
            curve["lib_size"] == L_SMALL,
            "rho"
        ].iloc[0]
    )

    rho_Lmax = float(
        curve.loc[
            curve["lib_size"] == L_MAX,
            "rho"
        ].iloc[0]
    )

    # ----------------------------------------
    # gain
    # ----------------------------------------

    gain = (
        rho_Lmax
        - rho_L10
    )

    # ----------------------------------------
    # convergence monotonicity
    # ----------------------------------------

    spearman_value, spearman_p = (
        spearmanr(
            L,
            rho_curve
        )
    )

    # ----------------------------------------
    # terminal significance
    # ----------------------------------------

    p_two_sided = (
        pearson_r_pvalue(
            rho_Lmax,
            N_TERMINAL
        )
    )

    ci_lower, ci_upper = (
        fisher_z_ci(
            rho_Lmax,
            N_TERMINAL,
            confidence=CI_LEVEL
        )
    )

    # ----------------------------------------
    # Frozen convergence rule
    # ----------------------------------------

    pass_gain = (
        gain > 0
    )

    pass_spearman = (
        spearman_value
        >= SPEARMAN_THRESHOLD
    )

    pass_p = (
        p_two_sided
        < P_THRESHOLD
    )

    pass_ci = (
        ci_lower
        > 0
    )

    converged = (
        pass_gain
        and pass_spearman
        and pass_p
        and pass_ci
    )

    causal_strength = (
        rho_Lmax
        if converged
        else 0.0
    )

    diagnostic_rows.append({
        "feature":
            feature,

        "rho_L10":
            rho_L10,

        "rho_Lmax":
            rho_Lmax,

        "gain":
            gain,

        "spearman":
            spearman_value,

        "spearman_p":
            spearman_p,

        "p_two_sided":
            p_two_sided,

        "CI_lower":
            ci_lower,

        "CI_upper":
            ci_upper,

        "pass_gain":
            pass_gain,

        "pass_spearman":
            pass_spearman,

        "pass_p":
            pass_p,

        "pass_CI":
            pass_ci,

        "converged":
            converged,

        "causal_strength":
            causal_strength,
    })


diag_df = pd.DataFrame(
    diagnostic_rows
)

# ============================================================
# 5. Normalize causal weights
# ============================================================

strength_sum = (
    diag_df[
        "causal_strength"
    ].sum()
)

if (
    not np.isfinite(strength_sum)
    or strength_sum <= 0
):
    raise RuntimeError(
        "No feature passed GCCM convergence. "
        "Do NOT create causal weights."
    )

diag_df[
    "causal_weight"
] = (
    diag_df[
        "causal_strength"
    ]
    / strength_sum
)

# ============================================================
# 6. Distance coefficients used later in GCCP
#
# We scale normalized weights by p = 7:
#
# gamma_k = p * w_k
#
# Therefore:
# mean gamma among all features = 1
#
# If all features had equal weights:
# gamma_k = 1
# and causal distance reduces to ordinary standardized EUC.
# ============================================================

P = len(
    FEATURES
)

diag_df[
    "gamma"
] = (
    P
    * diag_df[
        "causal_weight"
    ]
)

# ============================================================
# 7. Output
# ============================================================

print("\n" + "=" * 80)
print("PM2.5 GCCM DIAGNOSTICS")
print("=" * 80)

display(
    diag_df[
        [
            "feature",
            "rho_L10",
            "rho_Lmax",
            "gain",
            "spearman",
            "p_two_sided",
            "CI_lower",
            "CI_upper",
            "converged",
            "causal_strength",
            "causal_weight",
            "gamma",
        ]
    ]
)

print("\nStrength sum:")
print(
    strength_sum
)

print("\nWeight sum:")
print(
    diag_df[
        "causal_weight"
    ].sum()
)

print("\nGamma mean:")
print(
    diag_df[
        "gamma"
    ].mean()
)

print("\nConverged:")
print(
    diag_df[
        ["feature", "converged"]
    ]
)

# ============================================================
# 8. Ranking
# ============================================================

ranking_df = (
    diag_df[
        [
            "feature",
            "rho_Lmax",
            "causal_strength",
            "causal_weight",
            "gamma",
            "converged",
        ]
    ]
    .sort_values(
        "causal_weight",
        ascending=False
    )
    .reset_index(drop=True)
)

ranking_df[
    "rank"
] = (
    np.arange(
        1,
        len(ranking_df) + 1
    )
)

print("\n" + "=" * 80)
print("CAUSAL WEIGHT RANKING")
print("=" * 80)

display(
    ranking_df[
        [
            "rank",
            "feature",
            "rho_Lmax",
            "converged",
            "causal_strength",
            "causal_weight",
            "gamma",
        ]
    ]
)

# ============================================================
# 9. Direction diagnostic
#
# Save opposite terminal rho too.
# This is diagnostic only.
# It does NOT change the pre-specified causal direction.
# ============================================================

direction_rows = []

for feature in FEATURES:

    for direction in [
        "cause_xmap_effect",
        "effect_xmap_cause"
    ]:

        row = (
            mean_df[
                (mean_df["feature"] == feature)
                &
                (mean_df["direction"] == direction)
                &
                (mean_df["lib_size"] == L_MAX)
            ]
        )

        direction_rows.append({
            "feature":
                feature,

            "direction":
                direction,

            "rho_Lmax":
                float(
                    row["rho"].iloc[0]
                )
        })

direction_terminal_df = pd.DataFrame(
    direction_rows
)

direction_pivot = (
    direction_terminal_df
    .pivot(
        index="feature",
        columns="direction",
        values="rho_Lmax"
    )
    .reset_index()
)

direction_pivot[
    "effect_minus_cause_xmap"
] = (
    direction_pivot[
        "effect_xmap_cause"
    ]
    -
    direction_pivot[
        "cause_xmap_effect"
    ]
)

print("\n" + "=" * 80)
print("DIRECTION TERMINAL RHO DIAGNOSTIC")
print("=" * 80)

display(
    direction_pivot
)

# ============================================================
# 10. Save frozen PM2.5 causal weights
# ============================================================

DIAG_PATH = (
    GCCM_OUTPUT_DIR
    / "pm25_gccm_causal_diagnostics_v1.csv"
)

WEIGHT_PATH = (
    GCCM_OUTPUT_DIR
    / "pm25_gccm_causal_weights_v1.csv"
)

DIRECTION_PATH = (
    GCCM_OUTPUT_DIR
    / "pm25_gccm_direction_terminal_rho_v1.csv"
)

diag_df.to_csv(
    DIAG_PATH,
    index=False,
    encoding="utf-8-sig"
)

ranking_df.to_csv(
    WEIGHT_PATH,
    index=False,
    encoding="utf-8-sig"
)

direction_pivot.to_csv(
    DIRECTION_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("\nSaved:")
print(DIAG_PATH)
print(WEIGHT_PATH)
print(DIRECTION_PATH)

print("\n✅ STEP 12C 完成")
print("✅ PM2.5 GCCM causal weights 已冻结")
print("⚠️ Calibration / Validation / Frozen Test 均未参与 GCCM")

STEP 12C — PM2.5 GCCM CAUSAL WEIGHTS
Mean rows: 84
Raw rows : 69174

Frozen criteria:
gain > 0
Spearman >= 0.6
p < 0.05
CI lower > 0
Terminal N = 988

PM2.5 GCCM DIAGNOSTICS


,feature,rho_L10,rho_Lmax,gain,spearman,p_two_sided,CI_lower,CI_upper,converged,causal_strength,causal_weight,gamma
0,dem,0.036820,0.291035,0.254215,1.0,9.700903e-21,0.232894,0.347103,True,0.291035,0.143548,1.004839
1,sp,0.050695,0.314610,0.263915,1.0,3.867342e-24,0.257290,0.369724,True,0.314610,0.155176,1.086235
2,tp,0.016330,0.213571,0.197242,1.0,1.177208e-11,0.153244,0.272313,True,0.213571,0.105341,0.737385
3,blh,0.025070,0.252402,0.227331,1.0,7.992381e-16,0.193072,0.309892,True,0.252402,0.124493,0.871452
4,r,0.036799,0.275498,0.238699,1.0,1.146115e-18,0.216855,0.332159,True,0.275498,0.135885,0.951196
5,aod_sat,0.247000,0.582239,0.335240,1.0,9.940033e-91,0.539460,0.622020,True,0.582239,0.287181,2.010264
6,ndvi,0.000000,0.098078,0.098078,1.0,2.026066e-03,0.035930,0.159471,True,0.098078,0.048376,0.338629



Strength sum:
2.0274332109202944

Weight sum:
0.9999999999999998

Gamma mean:
0.9999999999999999

Converged:
   feature  converged
0      dem       True
1       sp       True
2       tp       True
3      blh       True
4        r       True
5  aod_sat       True
6     ndvi       True

CAUSAL WEIGHT RANKING


,rank,feature,rho_Lmax,converged,causal_strength,causal_weight,gamma
0,1,aod_sat,0.582239,True,0.582239,0.287181,2.010264
1,2,sp,0.314610,True,0.314610,0.155176,1.086235
2,3,dem,0.291035,True,0.291035,0.143548,1.004839
3,4,r,0.275498,True,0.275498,0.135885,0.951196
4,5,blh,0.252402,True,0.252402,0.124493,0.871452
5,6,tp,0.213571,True,0.213571,0.105341,0.737385
6,7,ndvi,0.098078,True,0.098078,0.048376,0.338629



DIRECTION TERMINAL RHO DIAGNOSTIC


direction,feature,cause_xmap_effect,effect_xmap_cause,effect_minus_cause_xmap
0,aod_sat,0.456984,0.582239,0.125256
1,blh,0.291053,0.252402,-0.038652
2,dem,0.236680,0.291035,0.054354
3,ndvi,0.036293,0.098078,0.061786
4,r,0.313875,0.275498,-0.038377
5,sp,0.250063,0.314610,0.064547
6,tp,0.229187,0.213571,-0.015616



Saved:
E:\GeoSIMCP_Reproduce\geoconformal-main\GCCP_PM2.5\outputs\gccm\pm25_gccm_causal_diagnostics_v1.csv
E:\GeoSIMCP_Reproduce\geoconformal-main\GCCP_PM2.5\outputs\gccm\pm25_gccm_causal_weights_v1.csv
E:\GeoSIMCP_Reproduce\geoconformal-main\GCCP_PM2.5\outputs\gccm\pm25_gccm_direction_terminal_rho_v1.csv

✅ STEP 12C 完成
✅ PM2.5 GCCM causal weights 已冻结
⚠️ Calibration / Validation / Frozen Test 均未参与 GCCM
